# Downstream check 2: PubMed-RCT sentence classification after tokenizer transplant
**Why this test:** the PubMedQA test (exp5) was at the majority-class rate for GPT-2 small, so it could not separate the methods. This one is easier to learn (5 classes: BACKGROUND, OBJECTIVE, METHODS, RESULTS, CONCLUSIONS; fine-tuned models reach roughly 80-90%) and depends on biomedical vocabulary, which is what the transplant changes.
**Setup:** PubMed 20k RCT (Dernoncourt & Lee, 2017), downloaded from its GitHub repository. Fixed random sample: 2000 training sentences, 2000 test sentences (seed 0), a LOW-DATA regime so the initialization is not washed out. Each model gets the donor's tokenizer, then is fine-tuned (mean-pooled hidden state + a fresh 5-way linear head, identical initialization for every method at the same seed). Accuracy is measured after EACH epoch, so the early-learning regime is visible.
**Rows:** `own tokenizer (reference)` = unchanged base model with its own tokenizer; `mean-fill` = all missing rows set to the shared mean (a floor); then the transplant methods (official OMP-64 as shipped, official OMP-64 on centred inputs, subword-mean, FOCUS, our calibrated ridge).
Turn Internet and GPU on. Run top to bottom. Every run is saved when finished, so a rerun resumes. Time (rough estimate, untested): about 1.5-2 hours for the two GPT-2 pairs.

In [ ]:
%pip install -q mergekit deepfocus datasets scikit-learn

In [ ]:
import re, time, gc, random, pickle, subprocess, shutil, traceback, os, json
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: this will be very slow)")

## Config

In [ ]:
N_ABSTRACTS = 4000; TRAIN_FRAC = 0.7
N_CAL = 150                 # calibration sentences per text type (domain, general), from the training part
N_EVAL = 400                # evaluation sentences per text type
GRID = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5]
OMP_KS = (64, 32)           # 64 is the tool's default and the paper's setting
EXTRA_OFFICIAL = [("subword-mean", ["-a", "subword", "-s", "mean"]), ("common-interp", ["-a", "common_interpolation"]),
                  ("john-hewitt", ["-a", "john_hewitt"]), ("mean", ["-a", "mean"])]
EVAL_BATCH = 8
FOCUS_MIN_COUNT = 10        # fastText minimum token count inside FOCUS (the package default)
FOCUS_TEXT_MAX = 60000      # sentences used to train FOCUS's auxiliary fastText model (training text only)
DOMAIN_TXT = None; USE_WIKI = True
OUT = Path("official_results"); OUT.mkdir(exist_ok=True)
WORK = Path("ts_work"); WORK.mkdir(exist_ok=True)
PAIRS = [   # tied-embedding bases that mergekit-tokensurgeon supports (it has no OPT or GPT-Neo architecture file). The base keeps its weights and gets the donor's tokenizer.
    dict(name="GPT-2 <- Pythia tokenizer",         base="openai-community/gpt2",        donor="EleutherAI/pythia-160m"),
    dict(name="GPT-2 <- Qwen tokenizer",           base="openai-community/gpt2",        donor="Qwen/Qwen2.5-0.5B"),
    dict(name="GPT-2 medium <- Pythia tokenizer",  base="openai-community/gpt2-medium", donor="EleutherAI/pythia-160m"),
    dict(name="Qwen2.5-0.5B <- GPT-2 tokenizer",   base="Qwen/Qwen2.5-0.5B",            donor="openai-community/gpt2"),
]

In [ ]:
# Downstream configuration (overrides the PAIRS of the config cell above)
PAIRS = [
    dict(name="GPT-2 <- Pythia tokenizer",       base="openai-community/gpt2", donor="EleutherAI/pythia-160m",  seeds=(0, 1, 2, 3, 4), run=True),
    dict(name="GPT-2 <- Qwen tokenizer",         base="openai-community/gpt2", donor="Qwen/Qwen2.5-0.5B",       seeds=(0, 1, 2),       run=True),    # 109k missing tokens: the hardest pair, initialization matters most here
    dict(name="Qwen2.5-0.5B <- GPT-2 tokenizer", base="Qwen/Qwen2.5-0.5B",     donor="openai-community/gpt2",  seeds=(0, 1),          run=False),   # slow (about 2 hours); switch on if time allows
]
USE_FOCUS = True
N_TRAIN = 2000; N_TEST = 2000
EPOCHS = 3; LR = 2e-5; HEAD_LR = 1e-3; BATCH = 16; MAX_LEN = 96
EMB = Path("emb_cache"); EMB.mkdir(exist_ok=True)
RES = Path("downstream_rct_results"); RES.mkdir(exist_ok=True)
RCT_LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
RCT_URL = "https://raw.githubusercontent.com/Franck-Dernoncourt/pubmed-rct/master/PubMed_20k_RCT/{}.txt"

## Text (calibration text and FOCUS text, as in exp4)

In [ ]:
def split_sents(text):
    text = " ".join(text.split())
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if 8 <= len(s.split()) <= 60]

def get_domain():
    out = []
    if DOMAIN_TXT: out = [l.strip() for l in open(DOMAIN_TXT, encoding="utf-8") if l.strip()][:N_ABSTRACTS]
    else:
        from datasets import load_dataset
        for cfg, stream in (("pqa_artificial", True), ("pqa_labeled", False)):
            try:
                for ex in load_dataset("qiaojin/PubMedQA", cfg, split="train", streaming=stream):
                    out.append(" ".join(ex["context"]["contexts"]))
                    if len(out) >= N_ABSTRACTS: break
                if out: print(f"domain text: PubMedQA/{cfg}, {len(out)} abstracts"); break
            except Exception as e: print(f"PubMedQA/{cfg} failed: {type(e).__name__}: {str(e)[:150]}")
    if not out: raise RuntimeError("Could not load the domain text. Turn Internet on, or set DOMAIN_TXT.")
    return out

def get_wiki():
    if not USE_WIKI: return []
    try:
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        s = split_sents(" ".join(x for x in ds["text"][:30000] if x.strip() and not x.startswith(" =")))
        print(f"general text: WikiText-2, {len(s)} sentences"); return s
    except Exception as e: print(f"WikiText-2 not available ({type(e).__name__})"); return []

def make_text():
    ab = get_domain(); n = int(len(ab) * TRAIN_FRAC)
    tr = [s for a in ab[:n] for s in split_sents(a)]; ev = [s for a in ab[n:] for s in split_sents(a)]
    wk = get_wiki(); rng = random.Random(0)
    smp = lambda L, k: rng.sample(L, min(k, len(L)))
    T = dict(cal_dom=smp(tr, N_CAL), eval_dom=smp(ev, N_EVAL), cal_gen=smp(wk[:len(wk) // 2], N_CAL) if wk else [], eval_gen=smp(wk[len(wk) // 2:], N_EVAL) if wk else [])
    T["train_text"] = tr[:3000] + (wk[:len(wk) // 2][:3000] if wk else [])
    T["fasttext_text"] = (tr + (wk[:len(wk) // 2] if wk else []))[:FOCUS_TEXT_MAX]
    print({k: len(v) for k, v in T.items()}); return T
TEXT = make_text()
FOCUS_JSONL = WORK / "focus_train.jsonl"
with open(FOCUS_JSONL, "w", encoding="utf-8") as fh:
    for t_ in TEXT["fasttext_text"]: fh.write(json.dumps({"text": t_}) + "\n")

## Scoring, ridge and tool helpers

In [ ]:
def load_lm(path):
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForCausalLM.from_pretrained(path).float().to(DEVICE).eval()
    return tok, m

@torch.no_grad()
def sent_nll(model, tok, sents):
    """Total NLL of each sentence under the model with its own tokenizer (the first token is predicted from a BOS token)."""
    V = len(tok); bos = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    ids = [tok.encode(s, add_special_tokens=False) for s in sents]; order = sorted(range(len(ids)), key=lambda i: len(ids[i])); out = np.zeros(len(ids))
    for b in range(0, len(order), EVAL_BATCH):
        idx = order[b:b + EVAL_BATCH]; L = max(len(ids[i]) for i in idx) + 1
        x = torch.full((len(idx), L), bos, dtype=torch.long); m = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx): x[r, 1:1 + len(ids[i])] = torch.tensor(ids[i]); m[r, :1 + len(ids[i])] = 1
        x, m = x.to(DEVICE), m.to(DEVICE)
        lg = model(input_ids=x, attention_mask=m).logits[:, :-1, :V].float()
        nl = F.cross_entropy(lg.reshape(-1, V), x[:, 1:].reshape(-1), reduction="none").view(len(idx), -1)
        out[idx] = (nl * m[:, 1:].float()).sum(1).cpu().numpy()
    return out

def score_all(model, tok):
    return {k: sent_nll(model, tok, TEXT[k]) for k in ("eval_dom", "eval_gen") if TEXT[k]}

def unit(x, eps=1e-12): return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

def fit_ridge(Dc, Rc, seed=0):
    s = float(np.linalg.norm(Dc, axis=1).mean()); X = (Dc / s).astype(np.float64); Y = Rc.astype(np.float64)
    perm = np.random.default_rng(seed).permutation(len(X)); nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]
    def solve(Xt, Yt, lam):
        A = Xt.T @ Xt; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]
        return np.linalg.solve(A, Xt.T @ Yt)
    best = None
    for lam in (1e-4, 1e-3, 1e-2, 1e-1, 1.0):
        cos = (unit(X[va] @ solve(X[tr], Y[tr], lam)) * unit(Y[va])).sum(1).mean()
        if best is None or cos > best[0]: best = (cos, lam)
    return solve(X, Y, best[1]), s, best[1]

def ridge_dirs(Dc, Rc, Dz_c):
    W, s, lam = fit_ridge(Dc, Rc); return unit((Dz_c / s) @ W), (Dz_c / s) @ W, lam

def set_rows(model, Z, rows):
    w = model.get_input_embeddings().weight
    with torch.no_grad(): w[torch.as_tensor(Z, device=w.device)] = torch.as_tensor(np.asarray(rows, np.float32), device=w.device)

def cal_objective(model, tok): return float(np.mean([sent_nll(model, tok, TEXT[k]).mean() for k in ("cal_dom", "cal_gen") if TEXT[k]]))

def calibrate(model, tok, Z, mu_r, rbar, dirs):
    best = None
    for c in GRID:
        set_rows(model, Z, mu_r + c * rbar * dirs); o = cal_objective(model, tok)
        if best is None or o < best[1]: best = (c, o)
    return best

def fp32_path(mid):
    """Local float32 copy of a model: some checkpoints (Qwen) are stored in bfloat16, and mergekit-tokensurgeon then fails with a dtype mismatch."""
    p = WORK / "fp32" / str(mid).strip("/").replace("/", "__")
    if not (p / "config.json").exists():
        tok = AutoTokenizer.from_pretrained(mid); m = AutoModelForCausalLM.from_pretrained(mid).float()
        p.mkdir(parents=True, exist_ok=True); m.save_pretrained(p); tok.save_pretrained(p); del m; gc.collect()
    return str(p)

def run_ts(base, donor, outdir, args):
    if Path(outdir).exists(): shutil.rmtree(outdir)
    cmd = ["mergekit-tokensurgeon", base, donor, str(outdir), "--allow-crimes"] + (["--cuda"] if DEVICE == "cuda" else []) + args
    t = time.time(); p = subprocess.run(cmd, capture_output=True, text=True); dt = time.time() - t
    if p.returncode != 0: raise RuntimeError("mergekit-tokensurgeon failed:\n" + (p.stdout + p.stderr)[-1500:])
    return dt

## Build the embedding tables of all methods (cached)

In [ ]:
def tag_of(name): return re.sub(r"\W+", "_", name).strip("_")
def msafe(name): return re.sub(r"\W+", "_", name).strip("_")

def emb_of(path, V):
    mm = AutoModelForCausalLM.from_pretrained(path).float(); E = mm.get_input_embeddings().weight.detach().cpu().numpy()[:V].copy(); del mm; gc.collect(); return E

def shifted_copy(src, dst, sub_in, S_ids=None):
    mdl = AutoModelForCausalLM.from_pretrained(src).float(); tk_ = AutoTokenizer.from_pretrained(src)
    with torch.no_grad():
        wi = mdl.get_input_embeddings().weight; wi -= torch.as_tensor(sub_in, dtype=wi.dtype)
        wo = mdl.get_output_embeddings().weight
        if wo.data_ptr() != wi.data_ptr() and S_ids is not None: wo -= wo[torch.as_tensor(S_ids)].mean(0, keepdim=True)
    if Path(dst).exists(): shutil.rmtree(dst)
    Path(dst).mkdir(parents=True, exist_ok=True); mdl.save_pretrained(dst); tk_.save_pretrained(dst); del mdl; gc.collect()

def build_embeddings(pair):
    """Build the embedding table of every transplant method for one pair (cached on disk). Returns {method: array (V, d)} and the folder of the zero model (base weights + donor tokenizer)."""
    base, donor = fp32_path(pair["base"]), fp32_path(pair["donor"]); tag = tag_of(pair["name"]); d = EMB / tag; zdir = WORK / f"zero_{tag}"
    if (d / "names.json").exists() and (zdir / "config.json").exists():
        names = json.load(open(d / "names.json")); return {n: np.load(d / f"{i}.npy") for i, n in enumerate(names)}, zdir
    run_ts(base, donor, zdir, ["-a", "zero"])
    tok, m = load_lm(zdir); V = len(tok)
    E0 = m.get_input_embeddings().weight.detach().float().cpu().numpy()[:V].copy(); nrm = np.linalg.norm(E0, axis=1)
    if m.get_output_embeddings().weight.data_ptr() != m.get_input_embeddings().weight.data_ptr(): raise RuntimeError("untied model: not handled")
    Z = np.where(nrm < 1e-8)[0]; S = np.where(nrm >= 1e-8)[0]; print(f"  {pair['name']}: vocabulary {V} | shared {len(S)} | missing {len(Z)}")
    dm = AutoModel.from_pretrained(donor); D = dm.get_input_embeddings().weight.detach().float().cpu().numpy()[:V]; del dm
    mu_r, mu_d = E0[S].mean(0), D[S].mean(0); Rc, Dc = E0[S] - mu_r, D[S] - mu_d; rbar = float(np.linalg.norm(Rc, axis=1).mean()); Dz_c = D[Z] - mu_d
    cnt = np.bincount([i for s in TEXT["train_text"] for i in tok.encode(s, add_special_tokens=False) if i < V], minlength=V); f_idx = np.argsort(-cnt[S], kind="stable")[:5000]
    out = {}
    Em = E0.copy(); Em[Z] = mu_r; out["mean-fill"] = Em
    objs, Es = {}, {}
    for rn, (Dx, Rx) in (("ridge[all]", (Dc, Rc)), ("ridge[freq5000]", (Dc[f_idx], Rc[f_idx]))):
        dirs, _, _ = ridge_dirs(Dx, Rx, Dz_c); c, o = calibrate(m, tok, Z, mu_r, rbar, dirs); set_rows(m, Z, mu_r + c * rbar * dirs)
        objs[rn] = o; Es[rn] = m.get_input_embeddings().weight.detach().float().cpu().numpy()[:V].copy()
    sel = min(objs, key=objs.get); out["ridge[sel] (ours, calibrated)"] = Es[sel]; print("  ridge selected by the training objective:", sel)
    del m; gc.collect(); torch.cuda.empty_cache()
    for nm, args in (("OMP-64 (official)", ["-a", "omp", "-k", "64"]), ("subword-mean (official)", ["-a", "subword", "-s", "mean"])):
        run_ts(base, donor, WORK / "out_t", args); out[nm] = emb_of(WORK / "out_t", V); shutil.rmtree(WORK / "out_t", ignore_errors=True); print("  built", nm)
    try:     # official OMP-64 on mean-centred copies of both models, receiver mean added back afterwards
        shifted_copy(base, WORK / "c_base", mu_r); shifted_copy(donor, WORK / "c_donor", mu_d, S_ids=S)
        run_ts(WORK / "c_base", WORK / "c_donor", WORK / "out_c", ["-a", "omp", "-k", "64"])
        Ec = emb_of(WORK / "out_c", V); Ec = Ec + mu_r; out["OMP-64 (official, centred inputs)"] = Ec.astype(np.float32); print("  built OMP-64 (official, centred inputs)")
    except Exception as e: print("  !!! centred OMP failed:", e); traceback.print_exc()
    finally:
        for p_ in ("out_c", "c_base", "c_donor"): shutil.rmtree(WORK / p_, ignore_errors=True)
    if USE_FOCUS:
        try:
            from deepfocus import FOCUS
            btok = AutoTokenizer.from_pretrained(base); src = AutoModelForCausalLM.from_pretrained(base).get_input_embeddings().weight.detach().float().cpu()
            Fo = FOCUS(target_tokenizer=tok, source_tokenizer=btok, source_embeddings=src, target_training_data_path=str(FOCUS_JSONL), fasttext_model_min_count=FOCUS_MIN_COUNT, device="cpu", verbosity="silent")
            out["FOCUS (official)"] = Fo.cpu().numpy().astype(np.float32)[:V]; print("  built FOCUS (official)"); del src
        except Exception as e: print("  !!! FOCUS failed:", e); traceback.print_exc()
    d.mkdir(parents=True, exist_ok=True); names = list(out.keys())
    for i, n in enumerate(names): np.save(d / f"{i}.npy", out[n].astype(np.float32))
    json.dump(names, open(d / "names.json", "w")); return out, zdir

## Data, fine-tuning and evaluation

In [ ]:
from sklearn.metrics import f1_score

def parse_rct(txt):
    out = []
    for line in txt.splitlines():
        if "\t" in line:
            lab, s = line.split("\t", 1)
            if lab in RCT_LABELS and s.strip(): out.append((s.strip(), RCT_LABELS.index(lab)))
    return out

def get_split(name):
    p = Path("rct_data") / f"{name}.txt"; p.parent.mkdir(exist_ok=True)      # you can also upload your own copy to rct_data/train.txt and rct_data/test.txt
    if not p.exists():
        import urllib.request
        urllib.request.urlretrieve(RCT_URL.format(name), p)
    return parse_rct(p.read_text(encoding="utf-8"))

def load_rct():
    tr, te = get_split("train"), get_split("test"); rng = random.Random(0)
    return rng.sample(tr, N_TRAIN), rng.sample(te, N_TEST)

def encode_all(tok, exs):
    bos = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    return [([bos] + tok.encode(s, add_special_tokens=False)[:MAX_LEN - 1], y) for s, y in exs]

def collate(batch, pad):
    L = max(len(s) for s, _ in batch); ids = torch.full((len(batch), L), pad, dtype=torch.long); mask = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (s, _) in enumerate(batch): ids[i, :len(s)] = torch.tensor(s); mask[i, :len(s)] = 1
    return ids.to(DEVICE), mask.to(DEVICE), torch.tensor([y for _, y in batch]).to(DEVICE)

def finetune_eval(model, tok, train, test, seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    pad = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)
    tr, te = encode_all(tok, train), encode_all(tok, test)
    model.to(DEVICE); model.config.use_cache = False
    if sum(p.numel() for p in model.parameters()) > 3e8: model.gradient_checkpointing_enable()
    torch.manual_seed(seed); head = torch.nn.Linear(model.get_input_embeddings().weight.shape[1], len(RCT_LABELS)).to(DEVICE)    # same head initialization for every method at this seed
    def logits(ids, mask):
        h = model.base_model(input_ids=ids, attention_mask=mask, use_cache=False)[0]; m = mask.unsqueeze(-1).to(h.dtype)
        return head(((h * m).sum(1) / m.sum(1)).float())
    opt = torch.optim.AdamW([dict(params=model.parameters(), lr=LR, weight_decay=0.01), dict(params=head.parameters(), lr=HEAD_LR, weight_decay=0.0)])
    steps = EPOCHS * int(np.ceil(len(tr) / BATCH)); warm = max(1, int(0.1 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (steps - s) / (steps - warm)))
    rng = random.Random(seed); gold = np.array([y for _, y in te]); epochs = []; last = []
    for ep in range(EPOCHS):
        model.train(); order = list(range(len(tr))); rng.shuffle(order)
        for b in range(0, len(order), BATCH):
            ids, mask, y = collate([tr[i] for i in order[b:b + BATCH]], pad)
            loss = F.cross_entropy(logits(ids, mask), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(head.parameters()), 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); last.append(loss.item())
        model.eval(); preds = []
        with torch.no_grad():
            for b in range(0, len(te), 64):
                ids, mask, _ = collate(te[b:b + 64], pad); preds += logits(ids, mask).argmax(1).cpu().tolist()
        preds = np.array(preds); epochs.append(dict(acc=float((preds == gold).mean()), f1=float(f1_score(gold, preds, average="macro")), preds=preds.tolist(), train_loss=float(np.mean(last[-20:]))))
    return dict(epochs=epochs, gold=gold.tolist())

## Run

In [ ]:
train_set, test_set = load_rct()
print("train", len(train_set), "| test", len(test_set), "| test label counts", {l: sum(y == i for _, y in test_set) for i, l in enumerate(RCT_LABELS)})

def free():
    gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

for pair in PAIRS:
    if not pair.get("run", True): continue
    tag = tag_of(pair["name"]); print("\n" + "#" * 20, pair["name"], "#" * 20)
    embs, zdir = build_embeddings(pair)
    methods = ["own tokenizer (reference)"] + list(embs.keys())
    for seed in pair["seeds"]:
        for mn in methods:
            f = RES / f"{tag}__{msafe(mn)}__seed{seed}.json"
            if f.exists(): continue
            try:
                t0 = time.time()
                if mn == "own tokenizer (reference)":
                    tok = AutoTokenizer.from_pretrained(pair["base"]); model = AutoModelForCausalLM.from_pretrained(pair["base"]).float()
                else:
                    tok = AutoTokenizer.from_pretrained(zdir); model = AutoModelForCausalLM.from_pretrained(zdir).float()
                    with torch.no_grad(): model.get_input_embeddings().weight[:len(embs[mn])] = torch.as_tensor(embs[mn])
                r = finetune_eval(model, tok, train_set, test_set, seed); r.update(pair=pair["name"], method=mn, seed=seed, secs=time.time() - t0)
                json.dump(r, open(f, "w")); print(f"  seed {seed} | {mn:<40} acc by epoch " + " ".join(f"{e['acc']:.3f}" for e in r["epochs"]) + f" | final macro-F1 {r['epochs'][-1]['f1']:.3f} | {r['secs']:.0f}s")
            except Exception as e: print(f"  !!! {mn} seed {seed} failed: {type(e).__name__}: {e}"); traceback.print_exc()
            finally:
                for v_ in ("model", "tok"): globals().pop(v_, None)
                free()

## Report

In [ ]:
import glob
rows = [json.load(open(f)) for f in sorted(glob.glob(str(RES / "*.json")))]
def boot_ci(d, n=2000):
    rng = np.random.default_rng(0); m = d[rng.integers(0, len(d), (n, len(d)))].mean(1); return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))
for pn in dict.fromkeys(r["pair"] for r in rows):
    R = [r for r in rows if r["pair"] == pn]; meths = list(dict.fromkeys(r["method"] for r in R)); gold = np.array(R[0]["gold"])
    maj = max(np.mean(gold == k) for k in range(len(RCT_LABELS)))
    print("\n" + "=" * 118 + f"\n{pn} | test {len(gold)} sentences | majority-class accuracy {maj:.3f}\n" + "=" * 118)
    print(f"{'method':<44}{'seeds':>6}{'acc ep1':>9}{'acc ep2':>9}{'acc ep3':>9}{'sd ep3':>8}{'F1 ep3':>8}")
    corr = {}
    for mn in meths:
        rr = [r for r in R if r["method"] == mn]; ne = min(len(r["epochs"]) for r in rr)
        acc = [np.array([r["epochs"][e]["acc"] for r in rr]) for e in range(ne)]
        corr[mn] = {e: np.mean([(np.array(r["epochs"][e]["preds"]) == gold).astype(float) for r in rr], axis=0) for e in (0, ne - 1)}
        print(f"{mn:<44}{len(rr):>6}" + "".join(f"{a.mean():>9.3f}" for a in acc) + f"{(acc[-1].std(ddof=1) if len(rr) > 1 else float('nan')):>8.3f}{np.mean([r['epochs'][ne-1]['f1'] for r in rr]):>8.3f}")
    for ref in ("OMP-64 (official, centred inputs)", "OMP-64 (official)"):
        if ref not in corr: continue
        for e, nm in ((0, "after epoch 1"), (max(corr[ref]), "after the last epoch")):
            print(f"\n  paired difference in accuracy against '{ref}', {nm} (positive = method better; seed-averaged, bootstrap over test sentences)")
            for mn in meths:
                if mn == ref or e not in corr[mn]: continue
                m_, lo, hi = boot_ci(corr[mn][e] - corr[ref][e]); print(f"    {mn:<44}{m_:+.4f} [{lo:+.4f},{hi:+.4f}]{' *' if lo > 0 or hi < 0 else ''}")